In [24]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

print("All imports successful!")


All imports successful!


In [ ]:
netbanking = pd.read_csv(r"C:\Users\Sneha\OneDrive\Desktop\RebuttalAI\Data\netbanking_unauthorized.csv")
upi = pd.read_csv(r"C:\Users\Sneha\OneDrive\Desktop\RebuttalAI\Data\upi_unauthorized.csv")
non_delivery = pd.read_csv(r"C:\Users\Sneha\OneDrive\Desktop\RebuttalAI\Data\non_delivery.csv")
#for trial
print(netbanking.shape)
print(upi.shape)
print(non_delivery.shape)


(1000, 15)
(1200, 15)
(1400, 15)


In [10]:
for name, df in {
    "Netbanking": netbanking,
    "UPI": upi,
    "Non-delivery": non_delivery
}.items():

    print("\n", "="*50)
    print(name)
    print("="*50)

    print(df.info())
    print("\nLabel distribution:")
    print(df["label"].value_counts())



Netbanking
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   case_id                      1000 non-null   str    
 1   reason_code                  1000 non-null   str    
 2   payment_rail                 1000 non-null   str    
 3   order_value                  1000 non-null   float64
 4   days_since_transaction       1000 non-null   int64  
 5   device_ip_match_history      1000 non-null   float64
 6   auth_flow_type               1000 non-null   str    
 7   customer_account_age_days    1000 non-null   int64  
 8   customer_past_order_count    1000 non-null   int64  
 9   customer_past_dispute_count  1000 non-null   int64  
 10  delivery_confirmed           0 non-null      float64
 11  tracking_available           0 non-null      float64
 12  merchant_comm_log_exists     1000 non-null   int64  
 13  refund_already_iss

In [ ]:
for name, df in {
    "Netbanking": netbanking,
    "UPI": upi,
    "Non-delivery": non_delivery
}.items():

    print("\n", "=" * 40)
    print(name)
    print("=" * 40)

    print("Missing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    #the null columns are misleading so dropping these columns


Netbanking
Missing values:
delivery_confirmed    1000
tracking_available    1000
dtype: int64

UPI
Missing values:
delivery_confirmed    1200
tracking_available    1200
dtype: int64

Non-delivery
Missing values:
device_ip_match_history    1400
auth_flow_type             1400
dtype: int64


In [17]:
print("Netbanking:")
print(netbanking.columns.tolist())

print("\nUPI:")
print(upi.columns.tolist())

print("\nNon-delivery:")
print(non_delivery.columns.tolist())


Netbanking:
['case_id', 'reason_code', 'payment_rail', 'order_value', 'days_since_transaction', 'device_ip_match_history', 'auth_flow_type', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'merchant_comm_log_exists', 'refund_already_issued', 'label']

UPI:
['case_id', 'reason_code', 'payment_rail', 'order_value', 'days_since_transaction', 'device_ip_match_history', 'auth_flow_type', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'merchant_comm_log_exists', 'refund_already_issued', 'label']

Non-delivery:
['case_id', 'reason_code', 'payment_rail', 'order_value', 'days_since_transaction', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'delivery_confirmed', 'tracking_available', 'merchant_comm_log_exists', 'refund_already_issued', 'label']


In [18]:
# Features and target

drop_columns = ["case_id", "reason_code", "payment_rail", "label"]

X_netbanking = netbanking.drop(columns=drop_columns)
y_netbanking = netbanking["label"]

X_upi = upi.drop(columns=drop_columns)
y_upi = upi["label"]

X_non_delivery = non_delivery.drop(columns=drop_columns)
y_non_delivery = non_delivery["label"]

In [19]:
print("Netbanking:", X_netbanking.shape, y_netbanking.shape)
print("UPI:", X_upi.shape, y_upi.shape)
print("Non-delivery:", X_non_delivery.shape, y_non_delivery.shape)

Netbanking: (1000, 9) (1000,)
UPI: (1200, 9) (1200,)
Non-delivery: (1400, 9) (1400,)


In [20]:
print("Netbanking features:")
print(X_netbanking.columns.tolist())

print("\nUPI features:")
print(X_upi.columns.tolist())

print("\nNon-delivery features:")
print(X_non_delivery.columns.tolist())

Netbanking features:
['order_value', 'days_since_transaction', 'device_ip_match_history', 'auth_flow_type', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'merchant_comm_log_exists', 'refund_already_issued']

UPI features:
['order_value', 'days_since_transaction', 'device_ip_match_history', 'auth_flow_type', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'merchant_comm_log_exists', 'refund_already_issued']

Non-delivery features:
['order_value', 'days_since_transaction', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'delivery_confirmed', 'tracking_available', 'merchant_comm_log_exists', 'refund_already_issued']


In [ ]:
# Training data and testing data is split in this

X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_netbanking,
    y_netbanking,
    test_size=0.20,
    random_state=42,
    stratify=y_netbanking
)

X_train_upi, X_test_upi, y_train_upi, y_test_upi = train_test_split(
    X_upi,
    y_upi,
    test_size=0.20,
    random_state=42,
    stratify=y_upi
)

X_train_nd, X_test_nd, y_train_nd, y_test_nd = train_test_split(
    X_non_delivery,
    y_non_delivery,
    test_size=0.20,
    random_state=42,
    stratify=y_non_delivery
)   

In [22]:
print("Netbanking:", X_train_nb.shape, X_test_nb.shape)
print("UPI:", X_train_upi.shape, X_test_upi.shape)
print("Non-delivery:", X_train_nd.shape, X_test_nd.shape)

Netbanking: (800, 9) (200, 9)
UPI: (960, 9) (240, 9)
Non-delivery: (1120, 9) (280, 9)


In [23]:
# Numerical features

numeric_nb_upi = [
    "order_value",
    "days_since_transaction",
    "device_ip_match_history",
    "customer_account_age_days",
    "customer_past_order_count",
    "customer_past_dispute_count",
    "merchant_comm_log_exists",
    "refund_already_issued"
]

numeric_nd = [
    "order_value",
    "days_since_transaction",
    "customer_account_age_days",
    "customer_past_order_count",
    "customer_past_dispute_count",
    "delivery_confirmed",
    "tracking_available",
    "merchant_comm_log_exists",
    "refund_already_issued"
]

categorical_nb_upi = ["auth_flow_type"]
categorical_nd = []

In [25]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [26]:
# Netbanking preprocessor
preprocessor_nb = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_nb_upi),
        ("cat", categorical_transformer, categorical_nb_upi)
    ]
)

# UPI preprocessor
preprocessor_upi = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_nb_upi),
        ("cat", categorical_transformer, categorical_nb_upi)
    ]
)

# Non-delivery preprocessor
preprocessor_nd = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_nd)
    ]
)

In [ ]:
# Model Pipelines (NETBANKING)

In [ ]:
# ***Logistic Regression pipeline*** 

In [27]:
# Logistic Regression - Netbanking

lr_nb = Pipeline(steps=[
    ("preprocessor", preprocessor_nb),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train the model
lr_nb.fit(X_train_nb, y_train_nb)

print("Netbanking Logistic Regression trained successfully!")

Netbanking Logistic Regression trained successfully!


In [28]:
# Predictions

y_pred_nb = lr_nb.predict(X_test_nb)
y_prob_nb = lr_nb.predict_proba(X_test_nb)[:, 1]

print("Predictions generated successfully!")

Predictions generated successfully!


In [29]:
# Evaluate the model

accuracy = accuracy_score(y_test_nb, y_pred_nb)
precision = precision_score(y_test_nb, y_pred_nb)
recall = recall_score(y_test_nb, y_pred_nb)
f1 = f1_score(y_test_nb, y_pred_nb)
roc_auc = roc_auc_score(y_test_nb, y_prob_nb)

print("Netbanking - Logistic Regression")
print("--------------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Netbanking - Logistic Regression
--------------------------------
Accuracy : 0.8500
Precision: 0.8870
Recall   : 0.8571
F1 Score : 0.8718
ROC-AUC  : 0.9087


In [ ]:
# ***Random Forest pipeline***

In [30]:
# Random Forest - Netbanking

rf_nb = Pipeline(steps=[
    ("preprocessor", preprocessor_nb),
    ("classifier", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

rf_nb.fit(X_train_nb, y_train_nb)

y_pred_rf_nb = rf_nb.predict(X_test_nb)
y_prob_rf_nb = rf_nb.predict_proba(X_test_nb)[:, 1]

print("Netbanking Random Forest trained successfully!")

Netbanking Random Forest trained successfully!


In [31]:
# Evaluate Random Forest

accuracy_rf_nb = accuracy_score(y_test_nb, y_pred_rf_nb)
precision_rf_nb = precision_score(y_test_nb, y_pred_rf_nb)
recall_rf_nb = recall_score(y_test_nb, y_pred_rf_nb)
f1_rf_nb = f1_score(y_test_nb, y_pred_rf_nb)
roc_auc_rf_nb = roc_auc_score(y_test_nb, y_prob_rf_nb)

print("Netbanking - Random Forest")
print("--------------------------")
print(f"Accuracy : {accuracy_rf_nb:.4f}")
print(f"Precision: {precision_rf_nb:.4f}")
print(f"Recall   : {recall_rf_nb:.4f}")
print(f"F1 Score : {f1_rf_nb:.4f}")
print(f"ROC-AUC  : {roc_auc_rf_nb:.4f}")

Netbanking - Random Forest
--------------------------
Accuracy : 0.8600
Precision: 0.8824
Recall   : 0.8824
F1 Score : 0.8824
ROC-AUC  : 0.9109


In [ ]:
# ***XGBoost pipeline**

In [32]:
# XGBoost - Netbanking

xgb_nb = Pipeline(steps=[
    ("preprocessor", preprocessor_nb),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_nb.fit(X_train_nb, y_train_nb)

y_pred_xgb_nb = xgb_nb.predict(X_test_nb)
y_prob_xgb_nb = xgb_nb.predict_proba(X_test_nb)[:, 1]

print("Netbanking XGBoost trained successfully!")

Netbanking XGBoost trained successfully!


In [33]:
# Evaluate XGBoost

accuracy_xgb_nb = accuracy_score(y_test_nb, y_pred_xgb_nb)
precision_xgb_nb = precision_score(y_test_nb, y_pred_xgb_nb)
recall_xgb_nb = recall_score(y_test_nb, y_pred_xgb_nb)
f1_xgb_nb = f1_score(y_test_nb, y_pred_xgb_nb)
roc_auc_xgb_nb = roc_auc_score(y_test_nb, y_prob_xgb_nb)

print("Netbanking - XGBoost")
print("--------------------")
print(f"Accuracy : {accuracy_xgb_nb:.4f}")
print(f"Precision: {precision_xgb_nb:.4f}")
print(f"Recall   : {recall_xgb_nb:.4f}")
print(f"F1 Score : {f1_xgb_nb:.4f}")
print(f"ROC-AUC  : {roc_auc_xgb_nb:.4f}")

Netbanking - XGBoost
--------------------
Accuracy : 0.8350
Precision: 0.8308
Recall   : 0.9076
F1 Score : 0.8675
ROC-AUC  : 0.9073
